In [1]:
# !pip install yfinance

In [2]:
import yfinance as yf
import pandas as pd
import os
from datetime import date, timedelta

# ================= PATHS =================
input_file = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\YahooFinanceTickers.csv"
output_folder = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"

effective_date = date.today()
if effective_date.weekday() == 5:  # Saturday -> Friday
    effective_date -= timedelta(days=1)
elif effective_date.weekday() == 6:  # Sunday -> Friday
    effective_date -= timedelta(days=2)

run_date = effective_date.strftime("%Y%m%d")
output_file = os.path.join(output_folder, f"{run_date}_YahooConsensus.csv")

if os.path.exists(output_file):
    os.remove(output_file)

tickers = pd.read_csv(input_file)["Yahoo_Ticker"].tolist()
today = effective_date.strftime("%Y-%m-%d")

# ================= SAFE HELPERS =================
def safe(d, k):
    try:
        v = d.get(k, 0)
        return 0 if pd.isna(v) else v
    except:
        return 0

def growth(est, base):
    try:
        return round(est / base - 1, 4) if base else 0
    except:
        return 0

def disp_max(avg, maxv):
    try:
        return round((maxv - avg) / avg, 4) if avg else 0
    except:
        return 0

def disp_min(avg, minv):
    try:
        return round((minv - avg) / avg, 4) if avg else 0
    except:
        return 0

def get_latest_fy(ticker):
    try:
        fin = ticker.income_stmt
        return pd.to_datetime(fin.columns[0]).year
    except:
        return None

# ================= FETCH BOTH BO & NS ALWAYS =================
def fetch_both_tickers(base):
    t_bo = t_ns = None
    try:
        t_bo = yf.Ticker(base + ".BO")
    except:
        pass
    try:
        t_ns = yf.Ticker(base + ".NS")
    except:
        pass
    return t_bo, t_ns

# ================= MERGE ESTIMATE ROWS =================
def merge_rows(primary, fallback):
    """Fill missing fields in primary using fallback."""
    if primary is None:
        return fallback
    if fallback is None:
        return primary

    merged = {}
    for key in ["avg", "low", "high", "yearAgoRevenue", "yearAgoEps", "numberOfAnalysts"]:
        pv = safe(primary, key)
        fv = safe(fallback, key)
        merged[key] = pv if pv not in [0, None] else fv
    return merged

# ================= MAIN =================
rows = []

for base_symbol in tickers:

    # Fetch both BO and NS tickers
    t_bo, t_ns = fetch_both_tickers(base_symbol)

    # Determine which one has better analyst coverage
    ticker = None
    if t_bo and not t_bo.earnings_estimate.empty:
        ticker = t_bo
    elif t_ns and not t_ns.earnings_estimate.empty:
        ticker = t_ns
    else:
        ticker = t_bo or t_ns

    latest_fy = None
    earnings_bo = earnings_ns = pd.DataFrame()
    revenue_bo = revenue_ns = pd.DataFrame()
    info_bo = info_ns = {}

    if t_bo:
        earnings_bo = t_bo.earnings_estimate
        revenue_bo = t_bo.get_revenue_estimate()
        info_bo = t_bo.info

    if t_ns:
        earnings_ns = t_ns.earnings_estimate
        revenue_ns = t_ns.get_revenue_estimate()
        info_ns = t_ns.info

    # Pick best info (prefer BO)
    info = info_bo if safe(info_bo, "currentPrice") != 0 else info_ns

    if ticker:
        latest_fy = get_latest_fy(ticker)

    current_price = safe(info, "currentPrice")
    tp_avg = safe(info, "targetMeanPrice")
    tp_min = safe(info, "targetLowPrice")
    tp_max = safe(info, "targetHighPrice")

    periods = ["0y", "+1y"]

    for i, period in enumerate(periods):

        fin_year = (
            f"FY{(latest_fy + i + 1) % 100}"
            if latest_fy else f"Y{i+1}"
        )

        # Extract BO + NS rows
        rev_bo = revenue_bo.loc[period] if period in revenue_bo.index else None
        rev_ns = revenue_ns.loc[period] if period in revenue_ns.index else None
        eps_bo = earnings_bo.loc[period] if period in earnings_bo.index else None
        eps_ns = earnings_ns.loc[period] if period in earnings_ns.index else None

        # Merge BO + NS
        rev_row = merge_rows(rev_bo, rev_ns)
        eps_row = merge_rows(eps_bo, eps_ns)

        # Revenue
        rev_avg = safe(rev_row, "avg") / 1e7
        rev_min = safe(rev_row, "low") / 1e7
        rev_max = safe(rev_row, "high") / 1e7

        # EPS
        eps_avg = safe(eps_row, "avg")
        eps_min = safe(eps_row, "low")
        eps_max = safe(eps_row, "high")

        # Year-ago
        year_ago_rev = safe(rev_row, "yearAgoRevenue") / 1e7
        year_ago_eps = safe(eps_row, "yearAgoEps")

        rows.append({
            "YAHOO Ticker": base_symbol,
            "Date": today,
            "FIN Yr": fin_year,

            "TP(Avg)": round(tp_avg),
            "TP(Min)": round(tp_min),
            "TP(Max)": round(tp_max),

            "Rev.(Avg)": round(rev_avg, 1),
            "Rev.(Min)": round(rev_min, 1),
            "Rev.(Max)": round(rev_max, 1),

            "EPS(Avg)": round(eps_avg, 2),
            "EPS(Min)": round(eps_min, 2),
            "EPS(Max)": round(eps_max, 2),

            "#Count": int(safe(rev_row, "numberOfAnalysts")),

            "Year Ago Rev.": round(year_ago_rev, 1),
            "Year Ago EPS": round(year_ago_eps, 2),

            "Rev_Growth_(Min)": growth(rev_min, year_ago_rev),
            "Rev_Growth_(Avg)": growth(rev_avg, year_ago_rev),
            "Rev_Growth_(Max)": growth(rev_max, year_ago_rev),

            "EPS_Growth_(Min)": growth(eps_min, year_ago_eps),
            "EPS_Growth_(Avg)": growth(eps_avg, year_ago_eps),
            "EPS_Growth_(Max)": growth(eps_max, year_ago_eps),

            "Rev_Max_Avg_Change": disp_max(rev_avg, rev_max),
            "Rev_Avg_Min_Change": disp_min(rev_avg, rev_min),

            "EPS_Max_Avg_Change": disp_max(eps_avg, eps_max),
            "EPS_Avg_Min_Change": disp_min(eps_avg, eps_min),

            "TP_Max_Avg_Change": disp_max(tp_avg, tp_max),
            "TP_Avg_Min_Change": disp_min(tp_avg, tp_min),

            "Current Price": round(current_price, 2),
            "TP_Appreciation_(Min)": growth(tp_min, current_price),
            "TP_Appreciation_(Avg)": growth(tp_avg, current_price),
            "TP_Appreciation_(Max)": growth(tp_max, current_price)
        })

# ================= SAVE =================
pd.DataFrame(rows).to_csv(output_file, index=False)
print("Consensus data saved to:", output_file)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CDSL.BO"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CDSL.BO"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BSE.BO"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BSE.BO"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: JCHAC.BO"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: JCHAC.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: JCHAC.BO"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":

Consensus data saved to: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260417_YahooConsensus.csv


In [3]:
# import os
# import pandas as pd

# # === PATH ===
# BASE_PATH = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"

# LATEST_FILE = os.path.join(BASE_PATH, "20260207_YahooConsensus.csv")

# # === NEW COLUMN ORDER (exactly as you provided) ===
# COLUMN_ORDER = [
#     "YAHOO Ticker","Date","FIN Yr","TP(Avg)","TP(Min)","TP(Max)",
#     "Rev.(Avg)","Rev.(Min)","Rev.(Max)",
#     "EPS(Avg)","EPS(Min)","EPS(Max)",
#     "#Count",
#     "Year Ago Rev.","Year Ago EPS",
#     "Rev_Growth_(Min)","Rev_Growth_(Avg)","Rev_Growth_(Max)",
#     "EPS_Growth_(Min)","EPS_Growth_(Avg)","EPS_Growth_(Max)",
#     "Rev_Max_Avg_Change","Rev_Avg_Min_Change",
#     "EPS_Max_Avg_Change","EPS_Avg_Min_Change",
#     "TP_Max_Avg_Change","TP_Avg_Min_Change",
#     "Current Price",
#     "TP_Appreciation_(Min)","TP_Appreciation_(Avg)","TP_Appreciation_(Max)"
# ]

# # === Load latest file (source of truth) ===
# latest_df = pd.read_csv(LATEST_FILE)

# # Create lookup for Year Ago Rev/EPS
# lookup = latest_df.set_index(["YAHOO Ticker", "FIN Yr"])[["Year Ago Rev.", "Year Ago EPS"]]

# # === Process all CSVs in folder ===
# for file in os.listdir(BASE_PATH):
#     if not file.endswith(".csv"):
#         continue
#     if file == "20260207_YahooConsensus.csv":
#         continue  # skip latest file

#     full_path = os.path.join(BASE_PATH, file)
#     print(f"Updating: {file}")

#     df = pd.read_csv(full_path)

#     # --- Ensure all new columns exist ---
#     for col in COLUMN_ORDER:
#         if col not in df.columns:
#             df[col] = None

#     # --- Update Year Ago Rev/EPS ---
#     for idx, row in df.iterrows():
#         ticker = row["YAHOO Ticker"]
#         fin_year = row["FIN Yr"]
    
#         # Case 1: FY26 → use latest file actuals
#         if fin_year == "FY26":
#             key = (ticker, fin_year)
#             if key in lookup.index:
#                 df.at[idx, "Year Ago Rev."] = lookup.loc[key, "Year Ago Rev."]
#                 df.at[idx, "Year Ago EPS"] = lookup.loc[key, "Year Ago EPS"]
    
#         # Case 2: FY27 → use FY26 Avg Rev/EPS from SAME FILE
#         elif fin_year == "FY27":
#             prev_row = df[(df["YAHOO Ticker"] == ticker) & (df["FIN Yr"] == "FY26")]
    
#             if not prev_row.empty:
#                 df.at[idx, "Year Ago Rev."] = prev_row.iloc[0]["Rev.(Avg)"]
#                 df.at[idx, "Year Ago EPS"] = prev_row.iloc[0]["EPS(Avg)"]

#     # --- Compute Growth Columns ---
#     df["Rev_Growth_(Min)"] = ((df["Rev.(Min)"] / df["Year Ago Rev."]) - 1).round(4)
#     df["Rev_Growth_(Avg)"] = ((df["Rev.(Avg)"] / df["Year Ago Rev."]) - 1).round(4)
#     df["Rev_Growth_(Max)"] = ((df["Rev.(Max)"] / df["Year Ago Rev."]) - 1).round(4)
    
#     df["EPS_Growth_(Min)"] = ((df["EPS(Min)"] / df["Year Ago EPS"]) - 1).round(4)
#     df["EPS_Growth_(Avg)"] = ((df["EPS(Avg)"] / df["Year Ago EPS"]) - 1).round(4)
#     df["EPS_Growth_(Max)"] = ((df["EPS(Max)"] / df["Year Ago EPS"]) - 1).round(4)
    
#     # --- Dispersion ---
#     df["Rev_Max_Avg_Change"] = (((df["Rev.(Max)"] - df["Rev.(Avg)"]) / df["Rev.(Avg)"])).round(4)
#     df["Rev_Avg_Min_Change"] = (((df["Rev.(Min)"] - df["Rev.(Avg)"]) / df["Rev.(Avg)"])).round(4)
    
#     df["EPS_Max_Avg_Change"] = (((df["EPS(Max)"] - df["EPS(Avg)"]) / df["EPS(Avg)"])).round(4)
#     df["EPS_Avg_Min_Change"] = (((df["EPS(Min)"] - df["EPS(Avg)"]) / df["EPS(Avg)"])).round(4)
    
#     df["TP_Max_Avg_Change"] = (((df["TP(Max)"] - df["TP(Avg)"]) / df["TP(Avg)"])).round(4)
#     df["TP_Avg_Min_Change"] = (((df["TP(Min)"] - df["TP(Avg)"]) / df["TP(Avg)"])).round(4)
    
#     # --- Appreciation ---
#     df["TP_Appreciation_(Min)"] = ((df["TP(Min)"] / df["Current Price"]) - 1).round(4)
#     df["TP_Appreciation_(Avg)"] = ((df["TP(Avg)"] / df["Current Price"]) - 1).round(4)
#     df["TP_Appreciation_(Max)"] = ((df["TP(Max)"] / df["Current Price"]) - 1).round(4)

#     # --- Reorder columns ---
#     df = df[COLUMN_ORDER]

#     # --- Save back ---
#     df.to_csv(full_path, index=False)

# print("All files updated successfully.")

In [9]:
import yfinance as yf
import pandas as pd

# -------- CHANGE TICKER HERE --------
ticker = yf.Ticker("ICICIPRULI.NS")

# -------- TARGET PRICE --------
info = ticker.info

tp_avg = info.get("targetMeanPrice")
tp_min = info.get("targetLowPrice")
tp_max = info.get("targetHighPrice")

print("=== TARGET PRICE ===")
print(f"TP: {round(tp_avg)} / ({round(tp_min)} - {round(tp_max)})\n")

# -------- EPS FORECASTS --------
eps_df = ticker.earnings_estimate

print("=== EPS FORECASTS ===")

# Yahoo periods: "0y", "+1y", "+2y", "+5y"
for period, label in {"+1y": "1‑yr ahead EPS", "+2y": "2‑yr ahead EPS"}.items():
    if period in eps_df.index:
        row = eps_df.loc[period]
        eps_avg = row.get("avg")
        eps_min = row.get("low")
        eps_max = row.get("high")

        print(f"{label}: {round(eps_avg)} / ({round(eps_min)} - {round(eps_max)})")
    else:
        print(f"{label}: Not available")


=== TARGET PRICE ===
TP: 693 / (595 - 885)

=== EPS FORECASTS ===
1‑yr ahead EPS: 13 / (9 - 17)
2‑yr ahead EPS: Not available
